# Temporal Point Process Data Preprocessing

In [1]:
import os
import sys
import json
import pickle
import requests
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

/data1/tsukuda/anaconda3/envs/TPP-LLM2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [3]:
data_folder = os.path.join('..', 'data')

## US Earthquakes

### Downloading Data

Download the US earthquake data from 2020-01-01 (inclusive) to 2024-01-01 (exclusive) to `data/raw/us_earthquake`.

In [4]:
from io import StringIO
from datetime import datetime, timedelta

In [5]:
def download_earthquake_data_chunk(start_time, end_time, region):
    """Download earthquake data for a given time chunk."""
    
    # USGS Earthquake API endpoint
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    
    # Query parameters
    params = {
        "format": "csv",           # Output format
        "starttime": start_time,    # Start date (YYYY-MM-DD)
        "endtime": end_time,        # End date (YYYY-MM-DD)
        # "minmagnitude": min_magnitude,  # Minimum magnitude
        # "maxmagnitude": max_magnitude,  # Maximum magnitude
        "minlatitude": region["minlatitude"],  # Min latitude of region
        "maxlatitude": region["maxlatitude"],  # Max latitude
        "minlongitude": region["minlongitude"],  # Min longitude of region
        "maxlongitude": region["maxlongitude"],  # Max longitude of region
    }
    
    # Send the request
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        print(f"Data downloaded successfully for {start_time} to {end_time}.")
        return response.content
    else:
        print(f"Failed to download data for {start_time} to {end_time}. HTTP Status Code: {response.status_code}.")
        return None

In [6]:
def download_earthquake_data(start_time, end_time, region, output_file, chunk_size):
    """Download earthquake data by splitting the request into smaller chunks."""
    
    # Convert start and end times to datetime objects
    start_date = datetime.strptime(start_time, "%Y-%m-%d")
    end_date = datetime.strptime(end_time, "%Y-%m-%d")
    
    # Initialize an empty DataFrame to store all results
    all_data = pd.DataFrame()
    
    # Loop through each month in the date range
    current_start = start_date
    while current_start < end_date:
        # Define the end of the current month
        current_end = (current_start + timedelta(days=chunk_size))
        if current_end > end_date:
            current_end = end_date
        
        # Download data for the current month
        data_chunk = download_earthquake_data_chunk(
            current_start.strftime("%Y-%m-%d"), current_end.strftime("%Y-%m-%d"), region)
        
        # If data is returned, append it to the main DataFrame
        if data_chunk:
            chunk_df = pd.read_csv(StringIO(data_chunk.decode('utf-8')))
            all_data = pd.concat([all_data, chunk_df], ignore_index=True)
        
        # Move to the next month
        current_start = current_end
    
    # Save the complete dataset to a CSV file
    all_data.to_csv(output_file, index=False)
    print(f"Data downloaded successfully and saved to {output_file}.")

In [ ]:
# Parameters for the earthquake search
start_time = "2020-01-01"    # Start date
end_time = "2024-01-01"      # End date
region = {
    "minlatitude": 24.6,     # Min latitude of the region
    "maxlatitude": 50.0,     # Max latitude
    "minlongitude": -125.0,  # Min longitude
    "maxlongitude": -65.0    # Max longitude
}
output_file = f"{data_folder}/raw/us_earthquake/us_earthquakes.csv"

# Download the earthquake data
download_earthquake_data(start_time, end_time, region, output_file, chunk_size=30)

Data downloaded successfully for 2020-01-01 to 2020-01-31.
Data downloaded successfully for 2020-01-31 to 2020-03-01.
Data downloaded successfully for 2020-03-01 to 2020-03-31.
Data downloaded successfully for 2020-03-31 to 2020-04-30.
Data downloaded successfully for 2020-04-30 to 2020-05-30.
Data downloaded successfully for 2020-05-30 to 2020-06-29.
Data downloaded successfully for 2020-06-29 to 2020-07-29.
Data downloaded successfully for 2020-07-29 to 2020-08-28.
Data downloaded successfully for 2020-08-28 to 2020-09-27.
Data downloaded successfully for 2020-09-27 to 2020-10-27.
Data downloaded successfully for 2020-10-27 to 2020-11-26.
Data downloaded successfully for 2020-11-26 to 2020-12-26.
Data downloaded successfully for 2020-12-26 to 2021-01-25.
Data downloaded successfully for 2021-01-25 to 2021-02-24.
Data downloaded successfully for 2021-02-24 to 2021-03-26.
Data downloaded successfully for 2021-03-26 to 2021-04-25.
Data downloaded successfully for 2021-04-25 to 2021-05-2

### Loading Data

In [ ]:
df_earthquakes = pd.read_csv(f'{data_folder}/raw/us_earthquake/us_earthquakes.csv')

In [11]:
df_earthquakes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318770 entries, 0 to 318769
Data columns (total 22 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   time             318770 non-null  object 
 1   latitude         318770 non-null  float64
 2   longitude        318770 non-null  float64
 3   depth            318770 non-null  float64
 4   mag              318702 non-null  float64
 5   magType          318701 non-null  object 
 6   nst              314522 non-null  float64
 7   gap              318756 non-null  float64
 8   dmin             317592 non-null  float64
 9   rms              318756 non-null  float64
 10  net              318770 non-null  object 
 11  id               318770 non-null  object 
 12  updated          318770 non-null  object 
 13  place            318770 non-null  object 
 14  type             318770 non-null  object 
 15  horizontalError  249436 non-null  float64
 16  depthError       318761 non-null  floa

In [12]:
pd.to_datetime(df_earthquakes.time).describe()

count                                 318770
mean     2021-09-22 19:38:25.566545664+00:00
min         2020-01-01 00:11:06.520000+00:00
25%      2020-09-01 17:12:45.875000064+00:00
50%      2021-07-20 21:06:33.011500032+00:00
75%      2022-09-09 08:06:21.096750080+00:00
max         2023-12-31 23:56:09.140000+00:00
Name: time, dtype: object

### Preprocessing Data

In [13]:
df_earthquakes = df_earthquakes[
    (df_earthquakes["type"] == "earthquake") & (df_earthquakes["status"] == "reviewed") 
    & (df_earthquakes['magType'] == 'ml')]
df_earthquakes = df_earthquakes.dropna(subset=['time', 'latitude', 'longitude', 'mag'])\
    .drop_duplicates(subset=['time', 'latitude', 'longitude', 'mag'], keep='first')
df_earthquakes["time"] = pd.to_datetime(df_earthquakes["time"])
df_earthquakes['coordinate'] = df_earthquakes.apply(
    lambda row: (round(row['latitude']), round(row['longitude'])), axis=1)

In [14]:
df_earthquakes['coordinate'].value_counts()

coordinate
(36, -118)    25859
(34, -117)    20197
(33, -116)    18415
(38, -118)    16278
(32, -104)     8230
              ...  
(46, -108)        1
(46, -77)         1
(48, -108)        1
(32, -119)        1
(45, -105)        1
Name: count, Length: 452, dtype: int64

### Getting Sequence IDs

In [15]:
df_earthquakes = df_earthquakes.sort_values(by=["coordinate", "time"]).reset_index(drop=True)

In [16]:
seq_ids = []
seq_count = 0
last_time = df_earthquakes.loc[0, 'time']
last_coord = df_earthquakes.loc[0, 'coordinate']
max_hours = 24

for index, row in tqdm(df_earthquakes.iterrows(), total=len(df_earthquakes)):
    if row["coordinate"] != last_coord:
        seq_count += 1
    elif (row["time"] - last_time).total_seconds() / 3600 > max_hours:
        seq_count += 1
    
    seq_ids.append(seq_count)
    last_time = row["time"]
    last_coord = row["coordinate"]

df_earthquakes["seq_id"] = seq_ids

  0%|          | 0/180669 [00:00<?, ?it/s]

In [17]:
df_earthquakes.head()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,net,id,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource,coordinate,seq_id
0,2022-03-16 21:16:41.547000+00:00,25.3946,-100.1989,14.2300,3.4,ml,NaN,87.0,0.828,0.76,us,us6000h57x,2022-05-27T14:55:29.040Z,"5 km SW of Santiago, Mexico",earthquake,5.100000,7.100000,0.115,10.0,reviewed,us,us,"(25, -100)",0
1,2021-06-28 05:17:49.488000+00:00,26.1210,-96.2028,10.0000,2.7,ml,NaN,233.0,2.400,0.24,us,us7000eiwr,2021-09-09T22:06:53.040Z,"96 km E of South Padre Island, Texas",earthquake,11.600000,2.000000,0.105,12.0,reviewed,us,us,"(26, -96)",1
2,2021-04-11 03:42:00.619000+00:00,28.3401,-103.3944,10.0000,3.2,ml,NaN,104.0,1.018,0.70,us,us6000e0qk,2021-06-19T21:25:01.040Z,"50 km NE of Hércules, Mexico",earthquake,1.600000,2.000000,0.115,20.0,reviewed,us,us,"(28, -103)",2
3,2020-11-30 05:55:29.645000+00:00,28.4180,-100.2390,27.1698,1.5,ml,5.0,151.0,0.800,0.30,tx,tx2020xmrs,2025-07-02T20:00:41.018Z,"12 km SE of El Indio, Texas",earthquake,4.998712,6.937260,0.100,4.0,reviewed,tx,tx,"(28, -100)",3
4,2020-12-01 05:38:52.982000+00:00,28.4040,-100.3240,2.8987,2.9,ml,17.0,136.0,0.800,0.50,tx,tx2020xoms,2025-07-02T19:58:01.163Z,"11 km NNE of Guerrero, Mexico",earthquake,5.755505,2.375032,0.200,16.0,reviewed,tx,tx,"(28, -100)",3


### Selecting Sequences

In [18]:
df_earthquakes["seq_id"].value_counts().describe()

count    25480.000000
mean         7.090620
std        185.939029
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max      20126.000000
Name: count, dtype: float64

In [19]:
earthquake_counts = df_earthquakes["seq_id"].value_counts()
earthquake_list = earthquake_counts[(earthquake_counts >= 5) & (earthquake_counts <= 30)].index
len(earthquake_list)

3070

In [20]:
df_earthquakes = df_earthquakes[df_earthquakes["seq_id"].isin(earthquake_list)]

In [21]:
df_earthquakes.groupby('seq_id')['time'].count().describe()

count    3070.000000
mean        9.812052
std         5.658310
min         5.000000
25%         6.000000
50%         8.000000
75%        12.000000
max        30.000000
Name: time, dtype: float64

### Setting Event Types

In [22]:
df_earthquakes['mag'].describe()

count    30123.000000
mean         1.145491
std          0.711164
min         -1.020000
25%          0.660000
50%          1.100000
75%          1.580000
max          5.200000
Name: mag, dtype: float64

In [23]:
df_earthquakes["type"] = "Small"
df_earthquakes.loc[df_earthquakes["mag"] >= 1, "type"] = "Medium"
df_earthquakes.loc[df_earthquakes["mag"] >= 2, "type"] = "Large"
df_earthquakes["type"].value_counts()

type
Medium    13471
Small     12969
Large      3683
Name: count, dtype: int64

### Saving Sequences

In [24]:
df_earthquakes.groupby('seq_id')['time'].count().describe()

count    3070.000000
mean        9.812052
std         5.658310
min         5.000000
25%         6.000000
50%         8.000000
75%        12.000000
max        30.000000
Name: time, dtype: float64

In [ ]:
def get_seq_splits(df, seq_col):
    seq_ids = df[seq_col].unique().tolist()
    seq_ids_train, seq_ids_val_test = train_test_split(seq_ids, train_size=0.8, random_state=0)
    seq_ids_val, seq_ids_test = train_test_split(seq_ids_val_test, train_size=0.5, random_state=0)
    seq_splits = {seq_id: 'train' for seq_id in seq_ids_train}
    seq_splits.update({seq_id: 'dev' for seq_id in seq_ids_val})
    seq_splits.update({seq_id: 'test' for seq_id in seq_ids_test})
    print(f'train: {len(seq_ids_train)} seqs, val: {len(seq_ids_val)} seqs, test: {len(seq_ids_test)} seqs')
    return seq_splits

In [27]:
earthquake_seq_splits = get_seq_splits(df=df_earthquakes, seq_col='seq_id')
len(earthquake_seq_splits)

train: 2456 seqs, val: 307 seqs, test: 307 seqs


3070

In [28]:
df_earthquakes

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,net,id,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource,coordinate,seq_id
72,2021-07-23 20:59:36.058000+00:00,28.531000,-98.650000,3.7960,2.20,ml,11.0,89.0,0.3000,0.10,tx,tx2021oims,2025-07-02T20:01:21.524Z,"12 km NW of Tilden, Texas",Large,1.333471,1.512266,0.100000,7.0,reviewed,tx,tx,"(29, -99)",60
73,2021-07-24 08:08:40.563000+00:00,28.524000,-98.640000,4.7700,3.10,ml,21.0,72.0,0.2000,0.20,tx,tx2021ojit,2025-07-02T20:01:27.887Z,"11 km NW of Tilden, Texas",Large,1.134063,1.392107,0.000000,10.0,reviewed,tx,tx,"(29, -99)",60
74,2021-07-24 08:15:02.507000+00:00,28.518000,-98.644000,5.2568,1.90,ml,14.0,73.0,0.2000,0.20,tx,tx2021ojiy,2025-07-02T19:59:43.457Z,"11 km NW of Tilden, Texas",Medium,1.198728,1.570664,0.000000,8.0,reviewed,tx,tx,"(29, -99)",60
75,2021-07-24 17:46:59.979000+00:00,28.544000,-98.655000,3.9685,2.20,ml,16.0,80.0,0.3000,0.20,tx,tx2021okbv,2025-07-02T20:01:41.552Z,"13 km NW of Tilden, Texas",Large,1.556873,1.368526,0.100000,10.0,reviewed,tx,tx,"(29, -99)",60
76,2021-07-24 19:50:22.935000+00:00,28.520000,-98.631000,6.7249,1.80,ml,11.0,77.0,0.2000,0.20,tx,tx2021okfx,2025-07-02T20:01:38.370Z,"10 km NW of Tilden, Texas",Medium,1.183281,1.681270,0.100000,7.0,reviewed,tx,tx,"(29, -99)",60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180581,2021-12-06 00:06:56.990000+00:00,48.715667,-119.485000,0.3600,1.27,ml,9.0,172.0,0.3715,0.27,uw,uw61803126,2021-12-06T19:56:59.240Z,"3 km WNW of Tonasket, Washington",Medium,0.780000,22.830000,0.200781,6.0,reviewed,uw,uw,"(49, -119)",25401
180582,2021-12-06 04:44:35.340000+00:00,48.702667,-119.477500,0.2200,2.36,ml,16.0,172.0,0.3576,0.09,uw,uw61803151,2022-02-12T23:06:09.040Z,"2 km W of Tonasket, Washington",Large,0.450000,10.750000,0.174363,8.0,reviewed,uw,uw,"(49, -119)",25401
180583,2021-12-06 07:14:55.560000+00:00,48.704333,-119.475500,0.3400,2.16,ml,11.0,173.0,0.3589,0.09,uw,uw61803171,2022-02-12T23:06:11.040Z,"2 km W of Tonasket, Washington",Large,0.590000,10.860000,0.272194,7.0,reviewed,uw,uw,"(49, -119)",25401
180584,2021-12-06 08:55:03.160000+00:00,48.701667,-119.492167,0.1300,2.41,ml,13.0,169.0,0.3594,0.12,uw,uw61803176,2022-02-12T23:06:12.040Z,"3 km W of Tonasket, Washington",Large,0.440000,12.980000,0.065283,7.0,reviewed,uw,uw,"(49, -119)",25401


In [29]:
def save_seqs(
    df: pd.DataFrame, seq_col: str, seq_splits: dict,
    time_col: str, time_unit: float, type_col: str,mag_col: str,depth_col: str, seq_folder: str):
    """
    Save event sequences
    """
    dim_process = df[type_col].nunique()
    type_text2id = {type_text: type_id for type_id, type_text in enumerate(df[type_col].unique())}
    type_id2text = {type_id: type_text for type_text, type_id in type_text2id.items()}
    type_id_col = f'{type_col}_id'
    df[type_id_col] = df[type_col].map(type_text2id)
    data = {'train': [], 'dev': [], 'test': []}
    print(f'type_id2text: {type_id2text}')
    
    for seq_id, group in tqdm(df.groupby(seq_col)):
        group = group.sort_values(by=time_col).reset_index()
        split = seq_splits[seq_id]
        init_time = group[time_col].min()
        pre_event_time = init_time
        event_seq = {
            'dim_process': dim_process,
            'seq_idx': len(data[split]),
            'seq_len': len(group),
            'time_since_start': [],
            'time_since_last_event': [],
            'type_event': [],
            'type_text': [],
            'magnitude':[],
            'depth':[]
        }
        
        for index, row in group.iterrows():
            event_time = pd.to_datetime(row[time_col])
            time_since_start = (event_time - init_time).total_seconds() / time_unit
            time_since_last_event = (event_time - pre_event_time).total_seconds() / time_unit
            event_seq['time_since_start'].append(time_since_start)
            event_seq['time_since_last_event'].append(time_since_last_event)
            event_seq['type_event'].append(row[type_id_col])
            event_seq['type_text'].append(row[type_col])
            event_seq['magnitude'].append(row[mag_col])
            event_seq['depth'].append(row[depth_col])
            pre_event_time = event_time
        
        data[split].append(event_seq)

    os.makedirs(seq_folder, exist_ok=True)
    for split in ['train', 'dev', 'test']:
        json_path = f'{seq_folder}/{split}.json'
        with open(json_path, 'w') as file:
            json.dump(data[split], file, indent=4)
        print(f'{split} saved to {json_path}')

In [ ]:
save_seqs(
    df=df_earthquakes, seq_col='seq_id', seq_splits=earthquake_seq_splits,
    time_col='time', time_unit=60*60*24, type_col='type',mag_col='mag',depth_col='depth',
    seq_folder='data/us_earthquake',
)

type_id2text: {0: 'Large', 1: 'Medium', 2: 'Small'}


  0%|          | 0/3070 [00:00<?, ?it/s]

train saved to data/us_earthquake3/train.json
dev saved to data/us_earthquake3/dev.json
test saved to data/us_earthquake3/test.json
